In [ ]:
using Pkg
Pkg.activate(".")
Pkg.develop(path="..")

using Revise

In [ ]:
isCuda = try
    success(`nvidia-smi`)
catch
    false
end

In [ ]:
if isCuda
    println("CUDA is available. Loading CUDA.jl...")
    using CUDA
end

In [ ]:
using bslLD, Plots, Statistics

    
isCuda &&bslLD.use_cuda!()

In [ ]:
mutable struct Diag
    rhoe::Vector
    rhoi::Vector
    fi ::Vector
    fe ::Vector
end
Diag() = Diag([], [], [], [])

function diags!(diags, fe, fi, rhoe, rhoi, grid)
    push!(diags.rhoe, copy(rhoe.data[:]))
    push!(diags.rhoi, copy(rhoi.data[:]))
    push!(diags.fi, copy(fi.data))
    push!(diags.fe, copy(fe.data))
end

function step!(fi, fe, grid, simTime)
    rhoe = bslLD.compute_density(fe, grid)
    rhoi = bslLD.compute_density(fi, grid)
    sol = bslLD.solve_fields(bslLD.Moments(rhoi-rhoe), grid, bslLD.PoissonFieldSolver(-1.0))
    simTime.fraction_dt = 0.5
    bslLD.advectV!(fe, grid, simTime, sol.E)
    bslLD.advectV!(fi, grid, simTime, sol.E)
    simTime.fraction_dt = 1.0
    bslLD.advectX!(fe, grid, simTime)
    bslLD.advectX!(fi, grid, simTime)
    sol = bslLD.solve_fields(bslLD.Moments(rhoi-rhoe), grid, bslLD.PoissonFieldSolver(-1.0))
    simTime.fraction_dt = 0.5
    bslLD.advectV!(fe, grid, simTime, sol.E)
    bslLD.advectV!(fi, grid, simTime, sol.E)

    simTime.fraction_dt = 1.0
    return rhoe, rhoi
end


In [ ]:
grid =  bslLD.Grid([0.0,-4.0],[20.0,4.0],[64,64],1, 0.0, 1)

simTime = bslLD.SimulationTime(0.001, 30)

# initFuncv(v)= exp(-(v+2)^2 / 2) / sqrt(2*pi)+ exp(-(v-2)^2 / 2) / sqrt(2*pi)
# initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi)
# initFuncx(x) = 1+ 0.000001 * rand()
fi = bslLD.Distribution(grid, 0.0001, m=1.0, q=1.0);
fe = bslLD.Distribution(grid, 0.0001, m=0.001, q=-1.0);

In [ ]:
diags = Diag()
while bslLD.continue_advection(simTime,true)
    rhoe, rhoi = step!(fi, fe, grid, simTime)
    diags!(diags, fe, fi, rhoe, rhoi, grid)
    bslLD.advance!(simTime)
end    

In [ ]:
rhoe2 = map(x->(x .- mean(x, dims=1))[8], diags.rhoe)
rhoi2 = map(x->(x .- mean(x, dims=1))[8], diags.rhoi)

plot(rhoe2, label="Variance of rhoe", xlabel="Time step", ylabel="Variance")
plot!(rhoi2, label="Variance of rhoi", xlabel="Time step", ylabel="Variance")  

In [ ]:
function plotFs(nt)
    dfe = diags.fe[nt] .- mean(diags.fe[nt], dims=1)
    dfi = diags.fi[nt] .- mean(diags.fi[nt], dims=1)
    p1 = heatmap(dfe, title="Electrons", xlabel="Velocity index", ylabel="Position index")
    p2 = heatmap(dfi, title="Ions", xlabel="Velocity index", ylabel="Position index")
    plot(p1, p2, layout=(1,2))
end

In [ ]:
plotFs(17000)

In [ ]:
locData = transpose(hcat(map(x-> x.-mean(x), Array.(diags.rho))...))


Nx, Ny = size(locData)
w = kaiser(Ny, 6)

windowed = locData .* w'        # broadcast along second dim (1 × Ny)

heatmap(log.(abs.(fft(windowed))[1:200,1:round(Int,Ny/2)]))

In [ ]:

function measureStep(backendFunc,grid)
    backendFunc()

    f = bslLD.Distribution(grid, 0.5,initFuncv=initFuncv, initFuncx=initFuncx);
    e = bslLD.empty_vectorfield(grid);

    [step!(f, grid, Diag()) for _ in 1:10]   

    @time [step!(f, grid, Diag()) for _ in 1:10]
end

grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,257,257],0.02,10000,1, 1.0, 1)

measureStep(() -> bslLD.use_cuda!(),grid)
measureStep(() -> bslLD.use_cpu!(), grid)


#   1.270276 seconds (278.26 k allocations: 8.469 MiB, 1.77% gc time, 64.35% compilation time)
# 247.798143 seconds (46.79 k allocations: 29.209 GiB, 84.31% gc time, 0.89% compilation time)
#   0.059636 seconds (60.53 k allocations: 2.844 MiB, 20.65% gc time, 61.27% compilation time)
#   0.126698 seconds (46.27 k allocations: 83.402 MiB, 28.70% compilation time)





In [ ]:
#   0.018241 seconds (22.21 k allocations: 1.417 MiB, 22.05% gc time)
#  16.719033 seconds (2.88 k allocations: 1.254 GiB, 88.55% gc time)


grid =  bslLD.Grid([0.0,-4.0],[60.0,4.0],[1024,1024],0.02,10000,1, 1.0, 1)

measureStep(() -> bslLD.use_cuda!(),grid)
measureStep(() -> bslLD.use_cpu!(), grid)



In [ ]:
grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,255,255],0.02,10000,1, 1.0, 1)
f = bslLD.Distribution(grid, 0.5);

plan = bslLD.AdvectionPlan(f, grid)


In [ ]:
size(f.data)

In [ ]:
bslLD.use_cpu!()

In [ ]:
function compare_allocations(backendFunc)
    backendFunc()
    grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,257,257],0.02,10000,1, 1.0, 1)
    f = bslLD.Distribution(grid, 0.5);
    e = bslLD.empty_vectorfield(grid);

    plan = bslLD.AdvectionPlan(f, grid)

    #Warmup
    bslLD.advectX!(f,grid, plan)
    bslLD.advectX!(f,grid)
    bslLD.advectV!(f,grid,e)
    bslLD.advectV!(f,grid,e, plan)

    println("Measuring allocations for ", typeof(f.data))
    println("AdvectX with plan:")
    @time bslLD.advectX!(f,grid, plan)
    println("AdvectX without plan:")
    @time bslLD.advectX!(f,grid)
    println("AdvectV with plan:")
    @time bslLD.advectV!(f,grid,e, plan)
    println("AdvectV without plan:")
    @time bslLD.advectV!(f,grid,e)

    println(" ")

    println("Check for type stability:")
    println("AdvectX with plan:")
    @code_warntype bslLD.advectX!(f,grid, plan)
    println("AdvectX without plan:")
    @code_warntype bslLD.advectX!(f,grid)
    println("AdvectV with plan:")
    @code_warntype bslLD.advectV!(f,grid,e, plan)
    println("AdvectV without plan:")
    @code_warntype bslLD.advectV!(f,grid,e)
end

In [ ]:
compare_allocations(() -> bslLD.use_cuda!())


In [ ]:
compare_allocations(() -> bslLD.use_cpu!())

In [ ]:
CUDA.@profile bslLD.advectX!(f, grid, plan)

In [ ]:
CUDA.@profile bslLD.advectV!(f, grid, e, plan)

In [ ]:
print("Size of f.data: ")
println(size(f.data))
CUDA.@profile bslLD.advectV!(f, grid, e, plan)

In [ ]:

bslLD.use_cuda!()

grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,257,257],0.02,10000,1, 1.0, 1)
f = bslLD.Distribution(grid, 0.5);
e = bslLD.empty_vectorfield(grid);

plan = bslLD.AdvectionPlan(f, grid)


@code_warntype bslLD._advect_x_planned!(f, grid, plan, CUDABackend())
@code_warntype bslLD._advect_v_planned!(f, grid, e, plan, CUDABackend())

In [ ]:
@allocated ntuple(d -> e[d].data, Val(2))
#6000

In [ ]:
arr = f.data

In [ ]:
@allocated kernel!(arr, ctx; ndrange=length(arr))

In [ ]:

bslLD.use_cpu!()

grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,257,257],0.02,10000,1, 1.0, 1)
f = bslLD.Distribution(grid, 0.5);
e = bslLD.empty_vectorfield(grid);

plan = bslLD.AdvectionPlan(f, grid)


@code_warntype bslLD._advect_x_planned!(f, grid, plan, bslLD.KernelAbstractions.CPU())
@code_warntype bslLD._advect_v_planned!(f, grid, e, plan, bslLD.KernelAbstractions.CPU())

In [ ]:
plan.backend